In [ ]:

import pandas as pd
import os


# **Load Data**

In [ ]:

df = pd.read_csv('WDICSV.csv', engine='python')

# Load country metadata (for Region and Income Group)
df_country = pd.read_csv('WDICountry.csv')

# Sanity check on load
print("Main dataset shape:", df.shape)
print("Country metadata shape:", df_country.shape)
















Main dataset shape: (396970, 70)
Country metadata shape: (264, 31)


In [ ]:
# Filtering health indicators

health_indicators = [
    'SP.DYN.LE00.IN',    # Life expectancy at birth, total
    'SH.DYN.MORT',       # Under-5 mortality rate
    'SH.STA.MMRT',       # Maternal mortality ratio
    'SH.XPD.CHEX.GD.ZS', # Health expenditure (% of GDP)
    'SP.DYN.IMRT.IN'     # Infant mortality rate

]

df_health = df[df['Indicator Code'].isin(health_indicators)].copy()

In [ ]:
# Merging region and income group

# Keep only relevant columns from country metadata
df_country_meta = df_country[['Country Code', 'Region', 'Income Group']].copy()

# Drop rows where Region is blank — these are aggregates
# (World, Arab World, income-group totals), not individual countries
df_country_meta = df_country_meta.dropna(subset=['Region'])

# Inner join keeps only real countries (drops aggregates automatically)
df_health = df_health.merge(df_country_meta, on='Country Code', how='inner')

print("\nFiltered + merged dataset shape:", df_health.shape)
print("Indicators included:", df_health['Indicator Name'].unique())



Filtered + merged dataset shape: (1085, 72)
Indicators included: ['Current health expenditure (% of GDP)'
 'Life expectancy at birth, total (years)'
 'Maternal mortality ratio (modeled estimate, per 100,000 live births)'
 'Mortality rate, infant (per 1,000 live births)'
 'Mortality rate, under-5 (per 1,000 live births)']


# **EDA**

In [ ]:
# Missing data check

year_cols_all = [str(y) for y in range(1960, 2026)]
recent_years = [str(y) for y in range(2015, 2024)]

# Missing values per indicator, across all years
print("\n--- Missing values per indicator (all years) ---")
missing_by_indicator = df_health.groupby('Indicator Name')[year_cols_all].apply(
    lambda x: x.isnull().sum().sum()
)
print(missing_by_indicator)

# Missing values per indicator, recent decade only
print("\n--- Missing values per indicator (2015-2023) ---")
missing_recent = df_health.groupby('Indicator Name')[recent_years].apply(
    lambda x: x.isnull().sum().sum()
)
print(missing_recent)

# Country coverage by income group
print("\n--- Countries per income group ---")
print(df_health.groupby('Income Group')['Country Name'].nunique())



--- Missing values per indicator (all years) ---
Indicator Name
Current health expenditure (% of GDP)                                   9760
Life expectancy at birth, total (years)                                  251
Maternal mortality ratio (modeled estimate, per 100,000 live births)    6756
Mortality rate, infant (per 1,000 live births)                          2492
Mortality rate, under-5 (per 1,000 live births)                         2461
dtype: int64

--- Missing values per indicator (2015-2023) ---
Indicator Name
Current health expenditure (% of GDP)                                   223
Life expectancy at birth, total (years)                                   0
Maternal mortality ratio (modeled estimate, per 100,000 live births)    207
Mortality rate, infant (per 1,000 live births)                          189
Mortality rate, under-5 (per 1,000 live births)                         189
dtype: int64

--- Countries per income group ---
Income Group
High income            86
Low 

In [ ]:
#  focus on 2015-2023 (recent decade)
# - Life expectancy has 0% missing in this window
# - Under-5/infant mortality, maternal mortality, health expenditure
#   are all under 12% missing -- acceptable for trend analysis
# - Physicians per 1,000 people was dropped entirely (see Step 2 note)
#   before this step, since 42% missing made it unusable

recent_years = [str(y) for y in range(2015, 2024)]

# Keep only the recent decade's year columns (plus ID columns)
id_cols = ['Country Name', 'Country Code', 'Indicator Name', 'Indicator Code',
           'Region', 'Income Group']
df_health_recent = df_health[id_cols + recent_years].copy()


In [ ]:
# Converting data to long format

# country-indicator-year, rather than one column per year
df_long = df_health_recent.melt(
    id_vars=id_cols,
    value_vars=recent_years,
    var_name='Year',
    value_name='Value'
)

df_long['Year'] = df_long['Year'].astype(int)

print("\nLong format shape:", df_long.shape)
print(df_long.head())
print("\nRemaining missing values:", df_long['Value'].isnull().sum())


Long format shape: (9765, 8)
  Country Name Country Code  \
0  Afghanistan          AFG   
1  Afghanistan          AFG   
2  Afghanistan          AFG   
3  Afghanistan          AFG   
4  Afghanistan          AFG   

                                      Indicator Name     Indicator Code  \
0              Current health expenditure (% of GDP)  SH.XPD.CHEX.GD.ZS   
1            Life expectancy at birth, total (years)     SP.DYN.LE00.IN   
2  Maternal mortality ratio (modeled estimate, pe...        SH.STA.MMRT   
3     Mortality rate, infant (per 1,000 live births)     SP.DYN.IMRT.IN   
4    Mortality rate, under-5 (per 1,000 live births)        SH.DYN.MORT   

                       Region Income Group  Year       Value  
0  Middle East & North Africa   Low income  2015   10.105348  
1  Middle East & North Africa   Low income  2015   62.270000  
2  Middle East & North Africa   Low income  2015  741.000000  
3  Middle East & North Africa   Low income  2015   64.000000  
4  Middle East & 

In [ ]:
# exporting clean data to csv file

df_long.to_csv('health_indicators_clean_long.csv', index=False)
print("\nExported: health_indicators_clean_long.csv")


Exported: health_indicators_clean_long.csv
